# 01 - Run Judges

Idempotent pipeline: download -> inspect schema -> stratified samples -> run both judges -> error sanity check.

**Every step in this notebook is safe to re-run.** Downloads and samples are cached to disk;
judge calls skip any post_id that already has a successful result. If a run is interrupted
(rate limit, kernel restart, etc.), just re-run the notebook from the top -- no wasted API
calls will be made on already-judged posts, and previously-errored rows will automatically
retry.

**Before trusting anything below**, check the output of the schema-inspection cell against
the column-name and label constants in `src/config.py`. The dataset card's column names are
a best guess, not a verified schema.


In [ ]:
import sys
from pathlib import Path

# Allow `import src...` when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src import config, dataset, prompts
from src.judge_schema import AssistantJudgeResult, PersonaJudgeResult
from src.categorizer import ResultStore, get_client, run_judge

## 1. Download posts (cached to `data/raw/posts.parquet`)

In [ ]:
df = dataset.download_posts()
print(f"loaded {len(df)} rows")
df.head()


## 2. Inspect schema -- VERIFY before proceeding

Compare the printed column names and value counts against `config.COL_POST_ID`,
`config.COL_CONTENT`, `config.COL_CATEGORY`, `config.COL_TOXICITY`, `config.CATEGORY_LABELS`,
and `config.TOXICITY_LABELS`. If they don't match, fix `src/config.py` before continuing --
everything downstream reads column names from there, not hardcoded strings.


In [ ]:
dataset.inspect_columns(df)

## 3. Stratified samples (cached to `data/processed/*.parquet`)

25 posts per content category (up to 9 categories = 225 posts max per sample).
Once cached, re-running this cell will NOT reshuffle the sample -- it loads the
existing parquet file.

In [ ]:
assistant_sample = dataset.get_or_create_sample(
    df,
    name=config.ASSISTANT_SAMPLE_NAME,
    toxicity=config.TOXICITY_SAFE,
)
print(f"assistant sample: {len(assistant_sample)} rows")
assistant_sample[config.COL_CATEGORY].value_counts().sort_index()

In [ ]:
persona_sample = dataset.get_or_create_sample(
    df,
    name=config.PERSONA_SAMPLE_NAME,
    toxicity=config.TOXICITY_MALICIOUS,
)
print(f"high-toxicity persona sample: {len(persona_sample)} rows")
persona_sample[config.COL_CATEGORY].value_counts().sort_index()


## 4. Run judges

Results are appended to `results/judged/<judge_name>.jsonl` as they complete. Safe to
interrupt (Ctrl-C / kernel restart) and re-run -- already-judged post_ids are skipped.


In [ ]:
client = get_client()

assistant_store = ResultStore(config.JUDGED_DIR / f"{config.ASSISTANT_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=assistant_store,
    rows=assistant_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.ASSISTANT_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_assistant_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AssistantJudgeResult,
    content_key=config.COL_CONTENT,
    category_key=config.COL_CATEGORY,
)
print(f"assistant judge: {len(assistant_store.seen_ids())} successful rows cached")


In [ ]:
persona_store = ResultStore(config.JUDGED_DIR / f"{config.PERSONA_JUDGE_NAME}.jsonl")

run_judge(
    client=client,
    store=persona_store,
    rows=persona_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.PERSONA_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_persona_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=PersonaJudgeResult,
    content_key=config.COL_CONTENT,
    category_key=config.COL_CATEGORY,
)
print(f"persona judge: {len(persona_store.seen_ids())} successful rows cached")


## 5. Error sanity check

Rows with an `"error"` key were not successfully judged and will be retried automatically
on the next run of this notebook. A nonzero count here after a full run usually means rate
limits or transient API errors -- just re-run cell 4 above.


In [ ]:
for name, store in [("assistant", assistant_store), ("persona", persona_store)]:
    rows = store.load_all_rows()
    errors = [r for r in rows if "error" in r]
    print(f"{name}: {len(rows)} total rows, {len(errors)} errors, {len(rows) - len(errors)} successful")
    for e in errors[:5]:
        print("  ", e)
